# LeetCode 1029 — Two City Scheduling bằng Python

Link đề: https://leetcode.com/problems/two-city-scheduling/ · Độ khó: **Medium** · Chủ đề: Array, Greedy, Sorting

Skeleton LeetCode:

```python
class Solution:
    def twoCitySchedCost(self, costs: List[List[int]]) -> int:
        ...
```

Ghi chú: đáp án là **một số nguyên duy nhất** (chi phí tối thiểu; có thể có nhiều cách phân chia cùng chi phí nhưng ta chỉ trả chi phí). Điểm cốt lõi: tổng chi phí `= Σ b_i + Σ_{i∈A} (a_i − b_i)`; chọn đúng `n` người đi thành phố `A` sao cho tổng `(a_i − b_i)` nhỏ nhất ⇒ **sắp xếp theo `a_i − b_i`**, `n` người đầu đi `A`, còn lại đi `B`. Oracle độc lập: **thử mọi cách chọn `n` người đi `A`**.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: `2n` người, mỗi người có chi phí bay tới `A` hoặc `B`; đúng `n` người tới mỗi thành phố; tối thiểu hoá tổng chi phí.
2. Nắm công cụ: sắp xếp theo khoá `a − b`, `heapq.nsmallest`, DP `dp[i][j]`, so sánh trao đổi (swap).
3. Đi từ **thử mọi tập `n` người đi `A`** → **sắp xếp theo `a − b` (tham lam)** → **DP theo (số người đã xét, số người đi `A`)** → **gửi hết về `B` rồi chọn `n` người "hoàn tiền" lớn nhất** → **tìm kiếm cục bộ bằng hoán đổi**.
3. Hiểu mấu chốt: `a_i − b_i` là "chi phí **tăng thêm** khi đổi người `i` từ `B` sang `A`"; đúng `n` người phải đổi, nên chọn `n` người có mức tăng nhỏ nhất (có thể âm).
4. Tránh lỗi kinh điển: sắp theo `a` (hay `b`, hay `min(a, b)`) thay vì `a − b`; bỏ qua ràng buộc cân bằng (cộng `min(a, b)`); sắp ngược chiều; chia sai `n`; so sánh tỉ số `a / b`.

## 2. Đề bài & ví dụ

Một công ty định phỏng vấn `2n` người. Cho mảng `costs`, `costs[i] = [aCost_i, bCost_i]`: chi phí đưa người `i` tới thành phố `A` là `aCost_i`, tới thành phố `B` là `bCost_i`.

Trả về **chi phí tối thiểu** để đưa mọi người tới một thành phố sao cho **đúng `n` người** tới mỗi thành phố.

**Ví dụ 1:** `costs = [[10,20],[30,200],[400,50],[30,20]]` → `110` (`10 + 30 + 50 + 20`).

**Ví dụ 2:** `costs = [[259,770],[448,54],[926,667],[184,139],[840,118],[577,469]]` → `1859`.

**Ví dụ 3:** `costs = [[515,563],[451,713],[537,709],[343,819],[855,779],[457,60],[650,359],[631,42]]` → `3086`.

**Ràng buộc:** `2 * n == costs.length`; `2 <= costs.length <= 100`; `costs.length` chẵn; `1 <= aCost_i, bCost_i <= 1000`.

In [1]:
import heapq
import itertools
import random
import time
from typing import List

ex2 = [[259, 770], [448, 54], [926, 667], [184, 139], [840, 118], [577, 469]]
ex3 = [[515, 563], [451, 713], [537, 709], [343, 819], [855, 779], [457, 60], [650, 359], [631, 42]]
examples = [([[10, 20], [30, 200], [400, 50], [30, 20]], 110), (ex2, 1859), (ex3, 3086), ([[10, 20], [10, 20]], 30), ([[5, 1], [1, 5]], 2)]
for c, want in examples:
    print(f"costs = {c[:4]}{'...' if len(c) > 4 else ''} -> {want}")

costs = [[10, 20], [30, 200], [400, 50], [30, 20]] -> 110
costs = [[259, 770], [448, 54], [926, 667], [184, 139]]... -> 1859
costs = [[515, 563], [451, 713], [537, 709], [343, 819]]... -> 3086
costs = [[10, 20], [10, 20]] -> 30
costs = [[5, 1], [1, 5]] -> 2


## 3. Kiến thức nền

### 3.1 Phân rã chi phí

Gọi `A` là tập `n` người đi thành phố `A` (còn lại đi `B`). Tổng chi phí

`cost(A) = Σ_{i∈A} a_i + Σ_{i∉A} b_i = Σ_i b_i + Σ_{i∈A} (a_i − b_i)`.

Số hạng đầu **không phụ thuộc** cách chọn `A`; ta chỉ cần chọn `A` có `n` phần tử để **tổng `(a_i − b_i)` nhỏ nhất**.

In [2]:
costs = [[10, 20], [30, 200], [400, 50], [30, 20]]
diffs = [a - b for a, b in costs]
print("a - b =", diffs, "| Σ b =", sum(b for a, b in costs))
chosen = sorted(range(4), key=lambda i: diffs[i])[:2]
print("chọn n = 2 người có a - b nhỏ nhất:", chosen, "->", sum(b for a, b in costs) + sum(diffs[i] for i in chosen))

a - b = [-10, -170, 350, 10] | Σ b = 290
chọn n = 2 người có a - b nhỏ nhất: [1, 0] -> 110


### 3.2 Chọn `n` phần tử có tổng nhỏ nhất

Chọn `n` trong `2n` số sao cho tổng nhỏ nhất ⇒ lấy `n` số **nhỏ nhất** (không cần quan tâm dấu). Sắp xếp rồi lấy `n` số đầu hoặc `heapq.nsmallest`.

### 3.3 Vai trò của ràng buộc "đúng `n` người"

Nếu bỏ ràng buộc, mỗi người chọn `min(a_i, b_i)` — thường **vi phạm** cân bằng (ví dụ `[[1, 100], [1, 100]]`: cả hai thích `A`). Ràng buộc buộc `n` người phải đi `B` dù đắt hơn; ta chọn những người "ít thiệt nhất" khi đổi.

## 4. Ý tưởng nền tảng / trực giác

`costs = [[10,20],[30,200],[400,50],[30,20]]`: `a − b = −10, −170, 350, 10`. Hai giá trị nhỏ nhất: `−170` (người 2) và `−10` (người 1) ⇒ hai người này đi `A` (chi phí `30` và `10`), hai người còn lại đi `B` (`50`, `20`). Tổng `110` ✓.

Người 3 có `a − b = 350` lớn: đi `A` thiệt `350` so với `B`; nên đi `B`. Người 2 có `−170`: đi `A` **lời** `170` so với `B`; nên đi `A`.

## 5. Brute force — thử mọi tập `n` người đi `A`

**Ý tưởng:** theo đúng định nghĩa: duyệt mọi tổ hợp `C(2n, n)` người đi `A`, tính tổng chi phí (cộng `a_i` cho `A`, `b_i` cho phần còn lại), lấy nhỏ nhất. `O(C(2n, n) · n)` — chỉ cho `2n ≲ 18`; làm **oracle**.

In [3]:
def sched_brute(costs):
    m = len(costs)
    n = m // 2
    best = None
    for A in itertools.combinations(range(m), n):
        inA = set(A)
        total = sum(costs[i][0] if i in inA else costs[i][1] for i in range(m))
        if best is None or total < best:
            best = total
    return best


for c, want in examples:
    if len(c) <= 8:
        assert sched_brute(c) == want, (c, sched_brute(c))
print([sched_brute(c) for c, _ in examples])

[110, 1859, 3086, 30, 2]


In [4]:
# Code nộp LeetCode cho Brute force (C(2n, n) tổ hợp: chỉ minh hoạ)
import itertools


class SolutionBrute:
    def twoCitySchedCost(self, costs: List[List[int]]) -> int:
        m = len(costs)
        best = float("inf")
        for A in itertools.combinations(range(m), m // 2):
            inA = set(A)
            best = min(best, sum(costs[i][0] if i in inA else costs[i][1] for i in range(m)))
        return best

**Độ phức tạp:** `O(C(2n, n) · n)` — mũ.

**Nút thắt:** ta thử cả tập hợp trong khi chi phí là **tổng độc lập** của từng người sau khi tách `Σ b_i`.

## 6. Bước đột phá

`cost(A) = Σ b_i + Σ_{i∈A}(a_i − b_i)`. Chọn `n` người có `a_i − b_i` nhỏ nhất: sắp xếp theo `a − b` rồi `n` người đầu đi `A`. `O(n log n)`.

## 7. Các cách tiếp theo

Ngoài Brute force (mục 5), ta có thêm 4 cách: **Cách 2** sắp xếp theo `a − b` (đáp án chính), **Cách 3** DP `dp[i][j]`, **Cách 4** gửi hết về `B` rồi chọn `n` "hoàn tiền" bằng `heapq`, **Cách 5** tìm kiếm cục bộ bằng hoán đổi.

### Cách 2 — Sắp xếp theo `a − b`

**Giải thích từng đoạn:**

1. `costs.sort(key=lambda c: c[0] - c[1])`.
2. `n = len(costs) // 2`; `n` người đầu đi `A` (cộng `c[0]`), `n` người sau đi `B` (cộng `c[1]`).

In [5]:
def sched_sort(costs):
    cs = sorted(costs, key=lambda c: c[0] - c[1])        # người "đi A lời nhất" lên đầu
    n = len(cs) // 2
    return sum(c[0] for c in cs[:n]) + sum(c[1] for c in cs[n:])


for c, want in examples:
    assert sched_sort(c) == want
print([sched_sort(c) for c, _ in examples])

[110, 1859, 3086, 30, 2]


In [6]:
# Code nộp LeetCode cho Cách 2
class SolutionSort:
    def twoCitySchedCost(self, costs: List[List[int]]) -> int:
        costs = sorted(costs, key=lambda c: c[0] - c[1])
        n = len(costs) // 2
        return sum(c[0] for c in costs[:n]) + sum(c[1] for c in costs[n:])

**Độ phức tạp:** `O(n log n)` thời gian, `O(n)` bộ nhớ (bản sao sắp xếp).

### Cách 3 — DP theo (số người đã xét, số người đi `A`)

**Ý tưởng:** `dp[i][j]` = chi phí nhỏ nhất khi đã xét `i` người đầu và đúng `j` người trong số đó đi `A` (`i − j` người đi `B`). `dp[i][j] = min(dp[i−1][j−1] + a_i, dp[i−1][j] + b_i)` với điều kiện `j ≤ n` và `i − j ≤ n`. Đáp án `dp[2n][n]`.

In [7]:
def sched_dp(costs):
    m = len(costs)
    n = m // 2
    INF = float("inf")
    dp = [[INF] * (n + 1) for _ in range(m + 1)]
    dp[0][0] = 0
    for i in range(1, m + 1):
        a, b = costs[i - 1]
        for j in range(0, min(i, n) + 1):
            if i - j > n:                                   # số người đi B đã vượt n
                continue
            best = INF
            if j > 0 and dp[i - 1][j - 1] != INF:
                best = dp[i - 1][j - 1] + a                 # người i đi A
            if i - 1 >= j and dp[i - 1][j] != INF:
                best = min(best, dp[i - 1][j] + b)          # người i đi B
            dp[i][j] = best
    return dp[m][n]


for c, want in examples:
    assert sched_dp(c) == want
print([sched_dp(c) for c, _ in examples])

[110, 1859, 3086, 30, 2]


In [8]:
# Code nộp LeetCode cho Cách 3
class SolutionDP:
    def twoCitySchedCost(self, costs: List[List[int]]) -> int:
        m = len(costs)
        n = m // 2
        INF = float("inf")
        dp = [[INF] * (n + 1) for _ in range(m + 1)]
        dp[0][0] = 0
        for i in range(1, m + 1):
            a, b = costs[i - 1]
            for j in range(min(i, n) + 1):
                if i - j > n:
                    continue
                best = INF
                if j > 0:
                    best = dp[i - 1][j - 1] + a
                if i - 1 >= j:
                    best = min(best, dp[i - 1][j] + b)
                dp[i][j] = best
        return dp[m][n]

**Độ phức tạp:** `O(n²)` thời gian và bộ nhớ (`(2n + 1) · (n + 1)` ô).

### Cách 4 — Gửi hết về `B`, chọn `n` người "đổi" có `a − b` nhỏ nhất

**Ý tưởng:** tính `total = Σ b_i`; sau đó phải đổi đúng `n` người sang `A`, mỗi người làm tổng thay đổi `a_i − b_i`. Chọn `n` giá trị nhỏ nhất bằng `heapq.nsmallest(n, diffs)` (không cần sắp xếp toàn bộ).

In [9]:
def sched_heap(costs):
    n = len(costs) // 2
    total_b = sum(b for a, b in costs)
    return total_b + sum(heapq.nsmallest(n, (a - b for a, b in costs)))


for c, want in examples:
    assert sched_heap(c) == want
print([sched_heap(c) for c, _ in examples])

[110, 1859, 3086, 30, 2]


In [10]:
# Code nộp LeetCode cho Cách 4
import heapq


class SolutionHeap:
    def twoCitySchedCost(self, costs: List[List[int]]) -> int:
        n = len(costs) // 2
        return sum(b for a, b in costs) + sum(heapq.nsmallest(n, (a - b for a, b in costs)))

**Độ phức tạp:** `O(n log n)` (với `nsmallest` thực tế `O(n log k)`), `O(n)` bộ nhớ.

### Cách 5 — Tìm kiếm cục bộ bằng hoán đổi

**Ý tưởng:** bắt đầu từ một cách chia bất kỳ (`n` người đầu đi `A`). Lặp: nếu tồn tại `i ∈ A`, `j ∈ B` mà hoán đổi hai người (i sang `B`, j sang `A`) **giảm** chi phí, thì thực hiện; dừng khi không cải thiện được. Mục 10 chứng minh: tại cực tiểu địa phương mọi `a_i − b_i` của `A` đều `≤` mọi `a_j − b_j` của `B`, nên đó là cực tiểu toàn cục.

In [11]:
def sched_swap(costs):
    m = len(costs)
    n = m // 2
    side = [True] * n + [False] * n                          # True: đi A
    improved = True
    while improved:
        improved = False
        for i in range(m):
            if not side[i]:
                continue
            for j in range(m):
                if side[j]:
                    continue
                # đổi i: A -> B và j: B -> A  => thay đổi = (b_i + a_j) - (a_i + b_j)
                if costs[i][1] + costs[j][0] < costs[i][0] + costs[j][1]:
                    side[i], side[j] = False, True
                    improved = True
                    break                                    # i đã chuyển sang B: sang người kế tiếp
    return sum(costs[i][0] if side[i] else costs[i][1] for i in range(m))


for c, want in examples:
    assert sched_swap(c) == want
print([sched_swap(c) for c, _ in examples])

[110, 1859, 3086, 30, 2]


In [12]:
# Code nộp LeetCode cho Cách 5 (O(n^2) mỗi vòng; vẫn nhỏ với 2n <= 100)
class SolutionSwap:
    def twoCitySchedCost(self, costs: List[List[int]]) -> int:
        m = len(costs)
        n = m // 2
        side = [True] * n + [False] * n
        improved = True
        while improved:
            improved = False
            for i in range(m):
                if not side[i]:
                    continue
                for j in range(m):
                    if not side[j] and costs[i][1] + costs[j][0] < costs[i][0] + costs[j][1]:
                        side[i], side[j] = False, True
                        improved = True
                        break
        return sum(costs[i][0] if side[i] else costs[i][1] for i in range(m))

**Độ phức tạp:** mỗi lần hoán đổi giảm chi phí ít nhất `1` (số nguyên) nên số lần hoán đổi hữu hạn (`≤ tổng − cực tiểu`); mỗi vòng quét `O(n²)`. Với `2n ≤ 100` và chi phí `≤ 1000` vẫn nhanh.

## 8. TRACE từng bước

**Trace Cách 2** với `costs = [[10,20],[30,200],[400,50],[30,20]]`:

| người | `[a, b]` | `a − b` |
|---|---|---|
| 1 | `[10, 20]` | −10 |
| 2 | `[30, 200]` | −170 |
| 3 | `[400, 50]` | 350 |
| 4 | `[30, 20]` | 10 |

Sắp xếp theo `a − b`: `[30,200]` (−170), `[10,20]` (−10), `[30,20]` (10), `[400,50]` (350). `n = 2`:

- đi `A`: `30 + 10 = 40`;
- đi `B`: `20 + 50 = 70`;
- tổng `110` ✓.

Hàm trace in từng bước:

In [13]:
def trace_sort(costs):
    cs = sorted(costs, key=lambda c: c[0] - c[1])
    n = len(cs) // 2
    print("sau khi sắp theo a - b:")
    total = 0
    for idx, (a, b) in enumerate(cs):
        city = "A" if idx < n else "B"
        cost = a if idx < n else b
        total += cost
        print(f"  [{a:3d},{b:3d}] a-b={a - b:4d} -> thành phố {city}, chi phí {cost}")
    print("=>", total)
    return total


assert trace_sort([[10, 20], [30, 200], [400, 50], [30, 20]]) == 110

sau khi sắp theo a - b:
  [ 30,200] a-b=-170 -> thành phố A, chi phí 30
  [ 10, 20] a-b= -10 -> thành phố A, chi phí 10
  [ 30, 20] a-b=  10 -> thành phố B, chi phí 20
  [400, 50] a-b= 350 -> thành phố B, chi phí 50
=> 110


**Trace "vì sao sắp theo `a − b` mà không theo `a`":** sắp theo `a` chỉ xét chi phí đi `A`, bỏ qua việc người còn lại đi `B` tốn bao nhiêu; ví dụ tìm tự động được ở mục 12C (và Lỗi 2 mục 13) cho thấy cách sắp theo `a` cho đáp án lớn hơn tối ưu.

## 9. Code nộp LeetCode

Chọn **Cách 2** (sắp xếp theo `a − b`). Giữ nguyên `class Solution`, `twoCitySchedCost(self, costs)`:

In [14]:
class Solution:
    def twoCitySchedCost(self, costs: List[List[int]]) -> int:
        costs = sorted(costs, key=lambda c: c[0] - c[1])   # a - b nhỏ: đi A "lời" hơn đi B
        n = len(costs) // 2
        return sum(c[0] for c in costs[:n]) + sum(c[1] for c in costs[n:])


print([Solution().twoCitySchedCost([c[:] for c in cs]) for cs, _ in examples])
assert all(Solution().twoCitySchedCost([c[:] for c in cs]) == w for cs, w in examples)

[110, 1859, 3086, 30, 2]


## 10. Vì sao đúng

**(a) Phân rã.** `cost(A) = Σ_i b_i + Σ_{i∈A}(a_i − b_i)`. Vì `Σ b_i` cố định, cực tiểu hoá `cost(A)` ⇔ cực tiểu hoá `Σ_{i∈A} d_i` với `d_i = a_i − b_i`, `|A| = n`.

**(b) Chọn `n` số nhỏ nhất.** Với `|A| = n` cố định, tổng `Σ_{i∈A} d_i` nhỏ nhất đạt khi `A` gồm `n` giá trị `d_i` nhỏ nhất (bất kỳ tập khác nếu có phần tử `i ∈ A`, `j ∉ A` với `d_i > d_j` thì đổi `i` bằng `j` làm tổng giảm hoặc không đổi). Sắp xếp theo `d` và lấy `n` đầu là một cách chọn như vậy; ranh giới hoà (các `d` bằng nhau) cho cùng tổng.

**(c) Cách 3.** `dp[i][j]` là chi phí tối thiểu của `i` người đầu với đúng `j` đi `A`; công thức chuyển dựa vào lựa chọn của người `i`. Điều kiện `j ≤ n` và `i − j ≤ n` bảo đảm không vượt sức chứa. `dp[2n][n]` là đáp án.

**(d) Cách 4.** Chính là công thức (a) với `n` giá trị nhỏ nhất lấy bằng `nsmallest`.

**(e) Cách 5 (cực tiểu địa phương là toàn cục).** Hoán đổi `i ∈ A` với `j ∈ B` thay đổi chi phí `(b_i + a_j) − (a_i + b_j) = d_j − d_i`. Hoán đổi cải thiện khi `d_j < d_i`. Tại điểm dừng không có hoán đổi cải thiện: với mọi `i ∈ A`, `j ∈ B`: `d_i ≤ d_j`. Khi đó mọi phần tử của `A` có `d` không lớn hơn mọi phần tử của `B`, tức `A` là một tập `n` giá trị `d` nhỏ nhất ⇒ tối ưu theo (b). Thuật toán dừng vì chi phí là số nguyên giảm nghiêm ngặt và bị chặn dưới.

**(f) Vì sao `min(a, b)` mỗi người sai.** Ràng buộc cân bằng buộc đúng `n` người đi mỗi thành phố; giải pháp `Σ min(a_i, b_i)` thường không cân bằng (ví dụ `[[1,100],[1,100]]`: cả hai đi `A`).

**(g) Kiểm chứng bằng máy.** Mọi cách khớp oracle `C(2n, n)` trên **mọi** mảng với `2n = 2` (chi phí `1..4`), `2n = 4` (chi phí `1..3`), `2n = 6` (chi phí `{1, 2}`), `6 000` mảng ngẫu nhiên `2n = 8`, và khớp DP trên mảng ngẫu nhiên tới `2n = 100` (mục 12).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Mọi tổ hợp | `O(C(2n,n)·n)` | `O(n)` | oracle `2n ≲ 18` |
| 2. Sắp xếp theo `a − b` | `O(n log n)` | `O(n)` | **đáp án chính** |
| 3. DP `dp[i][j]` | `O(n²)` | `O(n²)` | độc lập với tham lam |
| 4. `heapq.nsmallest` | `O(n log n)` | `O(n)` | không sắp xếp toàn bộ |
| 5. Hoán đổi cục bộ | `O(n²·số_hoán_đổi)` | `O(n)` | minh hoạ bất biến |

Đo thực tế (`2n = 100`):

In [15]:
def bench(fn, arg, repeat=100):
    best = 1e9
    for _ in range(3):
        t0 = time.perf_counter()
        for _ in range(repeat):
            fn(arg)
        best = min(best, (time.perf_counter() - t0) / repeat)
    return best


rng = random.Random(1029)
big = [[rng.randint(1, 1000), rng.randint(1, 1000)] for _ in range(100)]
for name, fn in [("sort", sched_sort), ("dp", sched_dp), ("heap", sched_heap), ("swap", sched_swap)]:
    print(f"{name:6s}: {bench(fn, big, repeat=3 if name == 'swap' else 100) * 1e6:9.1f} us -> {fn(big)}")

sort  :      12.0 us -> 31548


dp    :     657.0 us -> 31548
heap  :      22.1 us -> 31548


swap  :    2345.8 us -> 31548


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mảng nhỏ với oracle `C(2n, n)`, (E) tính chất / ngẫu nhiên (seed cố định), (F) giới hạn `2n = 100`.

### A. Ví dụ của đề

In [16]:
IMPLS = {
    "sort": sched_sort,
    "dp": sched_dp,
    "heap": sched_heap,
    "swap": sched_swap,
    "brute": sched_brute,
    "Solution": lambda c: Solution().twoCitySchedCost([x[:] for x in c]),
    "SolutionSort": lambda c: SolutionSort().twoCitySchedCost([x[:] for x in c]),
    "SolutionDP": lambda c: SolutionDP().twoCitySchedCost([x[:] for x in c]),
    "SolutionHeap": lambda c: SolutionHeap().twoCitySchedCost([x[:] for x in c]),
    "SolutionSwap": lambda c: SolutionSwap().twoCitySchedCost([x[:] for x in c]),
    "SolutionBrute": lambda c: SolutionBrute().twoCitySchedCost([x[:] for x in c]),
}
SLOW = {"brute", "SolutionBrute"}
fast = [n for n in IMPLS if n not in SLOW]


def check(costs, want=None, brute_m=14):
    if want is None and len(costs) <= brute_m:
        want = sched_brute(costs)
    snap = [c[:] for c in costs]
    results = {}
    for name in IMPLS:
        if name in SLOW and len(costs) > brute_m:
            continue
        results[name] = IMPLS[name](costs)
    assert costs == snap, "cài đặt làm hỏng đầu vào"
    assert len(set(results.values())) == 1, (costs[:4], results)
    got = next(iter(results.values()))
    if want is not None:
        assert got == want, (costs[:4], got, want)
    return got


for c, want in examples:
    check(c, want, brute_m=8)
print("ví dụ đề: brute (≤ 8 người) và", len(IMPLS), "cài đặt đúng, không sửa đầu vào")

ví dụ đề: brute (≤ 8 người) và 11 cài đặt đúng, không sửa đầu vào


### B. Edge case

- `2n = 2`: một người đi `A`, một người đi `B`: `min(a₁ + b₂, b₁ + a₂)`.
- Mọi người cùng chi phí: `n(a + b)`.
- Mọi người thích `A` hơn (`a < b`): vẫn phải `n` người đi `B`.
- Mọi người thích `B` hơn.
- Một người có chênh lệch cực lớn (bắt buộc đi thành phố rẻ).
- `a = b` với mọi người (đáp án `Σ a`).
- Hoà trong `a − b` (nhiều giá trị bằng nhau ở ranh giới).

In [17]:
edge = [
    [[1, 2], [3, 4]], [[5, 5], [5, 5]], [[1, 100], [1, 100]], [[100, 1], [100, 1]], [[1, 100], [1, 100], [1, 100], [1, 100]], [[1, 1000], [1000, 1], [500, 500], [1, 1]],
    [[7, 7]] * 6, [[1, 3], [2, 4], [3, 5], [4, 6]], [[10, 10], [20, 20], [30, 30], [40, 40]], [[3, 1], [3, 1], [3, 1], [3, 1], [3, 1], [3, 1]],
    [[10, 20], [30, 200], [400, 50], [30, 20]], [[1, 2], [2, 1], [1, 2], [2, 1]], [[1000, 1], [1, 1000]] * 3,
]
for c in edge:
    check(c)
print(len(edge), "edge case: mọi cài đặt khớp brute")
print("[[1,100]]*2 ->", Solution().twoCitySchedCost([[1, 100], [1, 100]]), "(một người buộc đi B: 1 + 100)")

13 edge case: mọi cài đặt khớp brute
[[1,100]]*2 -> 101 (một người buộc đi B: 1 + 100)


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [18]:
def bug_min_each(costs):                               # mỗi người chọn min(a, b), bỏ qua cân bằng
    return sum(min(a, b) for a, b in costs)


def bug_sort_by_a(costs):                              # sắp theo a
    cs = sorted(costs, key=lambda c: c[0])
    n = len(cs) // 2
    return sum(c[0] for c in cs[:n]) + sum(c[1] for c in cs[n:])


def bug_sort_by_b_desc(costs):                         # sắp theo b giảm dần
    cs = sorted(costs, key=lambda c: -c[1])
    n = len(cs) // 2
    return sum(c[0] for c in cs[:n]) + sum(c[1] for c in cs[n:])


def bug_sort_by_ratio(costs):                          # sắp theo tỉ số a / b
    cs = sorted(costs, key=lambda c: c[0] / c[1])
    n = len(cs) // 2
    return sum(c[0] for c in cs[:n]) + sum(c[1] for c in cs[n:])


def bug_reverse_diff(costs):                           # sắp theo a - b GIẢM dần
    cs = sorted(costs, key=lambda c: c[1] - c[0])
    n = len(cs) // 2
    return sum(c[0] for c in cs[:n]) + sum(c[1] for c in cs[n:])


def bug_wrong_split(costs):                            # chia lệch: n + 1 người đi A
    cs = sorted(costs, key=lambda c: c[0] - c[1])
    n = len(cs) // 2 + 1
    return sum(c[0] for c in cs[:n]) + sum(c[1] for c in cs[n:])


def bug_greedy_by_min(costs):                          # xếp theo min(a, b) rồi gán thành phố rẻ hơn tới khi đầy
    cs = sorted(costs, key=lambda c: min(c))
    n = len(cs) // 2
    ca = cb = total = 0
    for a, b in cs:
        if (a <= b and ca < n) or cb >= n:
            total += a
            ca += 1
        else:
            total += b
            cb += 1
    return total


BUGS = [bug_min_each, bug_sort_by_a, bug_sort_by_b_desc, bug_sort_by_ratio, bug_reverse_diff, bug_wrong_split, bug_greedy_by_min]
pool = [[list(p) for p in cs] for cs in itertools.product(itertools.product(range(1, 5), repeat=2), repeat=4)]
for bug in BUGS:
    found = next((c for c in pool if bug(c) != sched_brute(c)), None)
    assert found is not None, bug.__name__
    print(f"{bug.__name__:20s} sai trên {found}: ra {bug(found)}, đúng {sched_brute(found)}")

bug_min_each         sai trên [[1, 1], [1, 2], [1, 2], [1, 2]]: ra 4, đúng 5
bug_sort_by_a        sai trên [[1, 1], [1, 1], [1, 1], [1, 2]]: ra 5, đúng 4
bug_sort_by_b_desc   sai trên [[1, 1], [1, 1], [1, 1], [3, 2]]: ra 6, đúng 5
bug_sort_by_ratio    sai trên [[1, 1], [1, 2], [1, 2], [2, 4]]: ra 7, đúng 6
bug_reverse_diff     sai trên [[1, 1], [1, 1], [1, 1], [1, 2]]: ra 5, đúng 4
bug_wrong_split      sai trên [[1, 1], [1, 1], [2, 1], [2, 1]]: ra 5, đúng 4
bug_greedy_by_min    sai trên [[1, 1], [1, 1], [1, 1], [1, 2]]: ra 5, đúng 4


### D. Vét cạn mảng nhỏ

Oracle `sched_brute` (mọi tổ hợp `n` người đi `A`, cộng chi phí theo định nghĩa). Mọi mảng:

- `2n = 2`: chi phí `1..4` (`16² = 256` mảng),
- `2n = 4`: chi phí `1..3` (`9⁴ = 6 561` mảng),
- `2n = 6`: chi phí `{1, 2}` (`4⁶ = 4 096` mảng),
- `2n = 8`: `6 000` mảng ngẫu nhiên chi phí `1..5`.

In [19]:
t0 = time.perf_counter()
cnt = 0
main_impls = ["sort", "dp", "heap", "swap", "Solution", "SolutionSort", "SolutionDP", "SolutionHeap", "SolutionSwap"]
suites = [
    (2, range(1, 5)),
    (4, range(1, 4)),
    (6, range(1, 3)),
]
for m, vals in suites:
    types = [[a, b] for a in vals for b in vals]
    for combo in itertools.product(types, repeat=m):
        c = [list(p) for p in combo]
        want = sched_brute(c)
        for name in main_impls:
            assert IMPLS[name](c) == want, (name, c, IMPLS[name](c), want)
        cnt += 1
print(cnt, "mảng (2n = 2: chi phí 1..4; 2n = 4: 1..3; 2n = 6: {1,2}):", len(main_impls), "cài đặt khớp oracle C(2n,n);", round(time.perf_counter() - t0, 1), "s")
assert cnt == 16 ** 2 + 9 ** 4 + 4 ** 6
rng = random.Random(102901)
for _ in range(6000):
    c = [[rng.randint(1, 5), rng.randint(1, 5)] for _ in range(8)]
    want = sched_brute(c)
    for name in main_impls:
        assert IMPLS[name](c) == want, (name, c)
print("6000 mảng 2n = 8 (chi phí 1..5): khớp oracle;", round(time.perf_counter() - t0, 1), "s")

10913 mảng (2n = 2: chi phí 1..4; 2n = 4: 1..3; 2n = 6: {1,2}): 9 cài đặt khớp oracle C(2n,n); 0.4 s


6000 mảng 2n = 8 (chi phí 1..5): khớp oracle; 1.1 s


In [20]:
# brute và các class brute trên mẫu nhỏ để chắc chắn cả hai cài đặt oracle khớp nhau
rng = random.Random(102902)
for _ in range(800):
    m = rng.choice([2, 4, 6, 8, 10])
    c = [[rng.randint(1, 1000), rng.randint(1, 1000)] for _ in range(m)]
    assert sched_brute(c) == SolutionBrute().twoCitySchedCost(c) == sched_dp(c) == Solution().twoCitySchedCost(c)
print("800 mảng 2n ≤ 10, chi phí tới 1000: hai cài đặt brute khớp DP và tham lam")

800 mảng 2n ≤ 10, chi phí tới 1000: hai cài đặt brute khớp DP và tham lam


### E. Tính chất / ngẫu nhiên (seed cố định)

(1) Các cách khớp DP (oracle độc lập) với `2n` tới `100`; (2) **hoán vị** người không đổi đáp án; (3) **đổi chỗ cột `a` và `b`** không đổi đáp án (đối xứng thành phố); (4) cộng `c` vào mọi `a_i` tăng đáp án đúng `n·c`; cộng `c` vào mọi `b_i` cũng `n·c`; (5) nhân mọi chi phí với `k` nhân đáp án với `k`; (6) `đáp án ≥ Σ min(a_i, b_i)` và `≤ Σ max(a_i, b_i)`; (7) tồn tại cách chia cụ thể (từ sắp xếp) đạt đáp án và có đúng `n` người mỗi thành phố.

In [21]:
def assignment(costs):
    """Một cách chia đạt tối ưu: danh sách chỉ số đi A."""
    order = sorted(range(len(costs)), key=lambda i: costs[i][0] - costs[i][1])
    return set(order[:len(costs) // 2])


rng = random.Random(20240117)
t0 = time.perf_counter()
for it in range(1500):
    m = rng.choice([2, 4, 6, 10, 20, 50, 100])
    hi = rng.choice([3, 20, 1000])
    c = [[rng.randint(1, hi), rng.randint(1, hi)] for _ in range(m)]
    n = m // 2
    want = sched_dp(c)                                                                  # (1)
    for name in ("sort", "heap", "Solution", "SolutionSort", "SolutionHeap"):
        assert IMPLS[name](c) == want, (name, it)
    if m <= 50:
        assert sched_swap(c) == want
    sh = c[:]
    rng.shuffle(sh)
    assert Solution().twoCitySchedCost(sh) == want                                      # (2)
    assert Solution().twoCitySchedCost([[b, a] for a, b in c]) == want                  # (3)
    k = rng.randint(1, 50)
    assert Solution().twoCitySchedCost([[a + k, b] for a, b in c]) == want + n * k      # (4)
    assert Solution().twoCitySchedCost([[a, b + k] for a, b in c]) == want + n * k      # (4)
    assert Solution().twoCitySchedCost([[a * 3, b * 3] for a, b in c]) == 3 * want      # (5)
    assert sum(min(a, b) for a, b in c) <= want <= sum(max(a, b) for a, b in c)         # (6)
    A = assignment(c)
    assert len(A) == n and sum(c[i][0] if i in A else c[i][1] for i in range(m)) == want   # (7)
print("1500 mảng ngẫu nhiên (2n tới 100): 7 tính chất đúng;", round(time.perf_counter() - t0, 1), "s")

1500 mảng ngẫu nhiên (2n tới 100): 7 tính chất đúng; 0.6 s


### F. Giới hạn

`2n = 100`, chi phí `1..1000`: ngẫu nhiên, mọi người thích `A`, mọi người thích `B`, mọi `a_i = b_i`, chênh lệch cực lớn xen kẽ; các cách nhanh khớp nhau và khớp DP.

In [22]:
t0 = time.perf_counter()
rng = random.Random(102903)
cases = [
    [[rng.randint(1, 1000), rng.randint(1, 1000)] for _ in range(100)],
    [[1, 1000]] * 100, [[1000, 1]] * 100, [[500, 500]] * 100, [[1, 1000], [1000, 1]] * 50,
    [[rng.randint(1, 10), rng.randint(990, 1000)] for _ in range(100)], [[i % 1000 + 1, (i * 7) % 1000 + 1] for i in range(100)],
]
for c in cases:
    res = {name: IMPLS[name](c) for name in ("sort", "dp", "heap", "swap", "Solution", "SolutionSort", "SolutionDP", "SolutionHeap", "SolutionSwap")}
    assert len(set(res.values())) == 1, res
    print(f"  đáp án = {res['Solution']}")
assert Solution().twoCitySchedCost([[1, 1000]] * 100) == 50 * 1 + 50 * 1000
assert Solution().twoCitySchedCost([[500, 500]] * 100) == 100 * 500
print("2n = 100: mọi cài đặt khớp nhau và khớp đáp án dựng sẵn;", round(time.perf_counter() - t0, 2), "s")

  đáp án = 36981
  đáp án = 50050
  đáp án = 50050
  đáp án = 50000
  đáp án = 100
  đáp án = 49861
  đáp án = 12400
2n = 100: mọi cài đặt khớp nhau và khớp đáp án dựng sẵn; 0.06 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: mỗi người chọn `min(a, b)`

In [23]:
c = [[1, 100], [1, 100]]
print("bug:", bug_min_each(c), "| đúng:", Solution().twoCitySchedCost(c))
assert bug_min_each(c) == 2 and Solution().twoCitySchedCost(c) == 101

bug: 2 | đúng: 101


**Vì sao sai:** bỏ qua ràng buộc **đúng `n` người** mỗi thành phố; cả hai người đều thích `A` nhưng một người **phải** đi `B`.

### Lỗi 2: sắp theo `a` (hoặc theo `b`)

In [24]:
bad = next(c for c in pool if bug_sort_by_a(c) != sched_brute(c))
print("phản ví dụ:", bad, "| bug (theo a):", bug_sort_by_a(bad), "| đúng:", sched_brute(bad))
assert bug_sort_by_a(bad) != sched_brute(bad)

phản ví dụ: [[1, 1], [1, 1], [1, 1], [1, 2]] | bug (theo a): 5 | đúng: 4


**Vì sao sai:** chi phí "tăng thêm" khi đổi từ `B` sang `A` là `a − b` (so sánh **hai** chi phí của cùng một người), không phải `a` một mình.

### Lỗi 3: sắp xếp ngược chiều

In [25]:
bad = next(c for c in pool if bug_reverse_diff(c) != sched_brute(c))
print("phản ví dụ:", bad, "| bug:", bug_reverse_diff(bad), "| đúng:", sched_brute(bad))
assert bug_reverse_diff(bad) != sched_brute(bad)

phản ví dụ: [[1, 1], [1, 1], [1, 1], [1, 2]] | bug: 5 | đúng: 4


**Vì sao sai:** người đi `A` phải là người có `a − b` **nhỏ nhất**; sắp ngược chiều (`b − a` tăng dần = `a − b` giảm dần) chọn những người đi `A` thiệt nhất.

### Lỗi 4: tỉ số `a / b`

In [26]:
bad = next(c for c in pool if bug_sort_by_ratio(c) != sched_brute(c))
print("phản ví dụ:", bad, "| bug:", bug_sort_by_ratio(bad), "| đúng:", sched_brute(bad))
assert bug_sort_by_ratio(bad) != sched_brute(bad)

phản ví dụ: [[1, 1], [1, 2], [1, 2], [2, 4]] | bug: 7 | đúng: 6


**Vì sao sai:** mục tiêu là tổng **chênh lệch** (cộng), nên khoá phải là hiệu `a − b`; tỉ số không phản ánh số tiền tăng thêm.

### Lỗi 5: chia lệch `n`

In [27]:
c = [[10, 20], [30, 200], [400, 50], [30, 20]]
print("bug:", bug_wrong_split(c), "| đúng:", Solution().twoCitySchedCost(c))
assert bug_wrong_split(c) != 110

bug: 120 | đúng: 110


**Vì sao sai:** đúng `n = len(costs) // 2` người đi mỗi thành phố; `n + 1` vi phạm cân bằng.

### Lỗi 6 (bonus): gán theo `min(a, b)` rồi "vá" khi thành phố đầy

In [28]:
bad = next(c for c in pool if bug_greedy_by_min(c) != sched_brute(c))
print("phản ví dụ:", bad, "| bug:", bug_greedy_by_min(bad), "| đúng:", sched_brute(bad))
assert bug_greedy_by_min(bad) != sched_brute(bad)

phản ví dụ: [[1, 1], [1, 1], [1, 1], [1, 2]] | bug: 5 | đúng: 4


**Vì sao sai:** tham lam theo `min(a, b)` rồi dồn người dư sang thành phố kia không chọn đúng **ai** nên đi thành phố đắt hơn (người có chênh lệch ít nhất).

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "chia `2n` đối tượng thành hai nhóm đều nhau với chi phí khác nhau theo nhóm" → tách `Σ cost_B + Σ_{A}(cost_A − cost_B)` và chọn `n` chênh lệch nhỏ nhất.

**Mẹo:**

1. Viết tổng chi phí dưới dạng "cơ sở + chênh lệch".
2. Sắp xếp theo chênh lệch, `n` đầu đi `A`.
3. `heapq.nsmallest(n, ...)` khi không cần thứ tự đầy đủ.
4. DP `dp[i][j]` là oracle độc lập tốt.
5. Chứng minh bằng "đổi chỗ" (exchange argument).
6. Khi số người đi mỗi nhóm khác nhau (`k` và `m − k`) vẫn dùng cách này với `k` chênh lệch nhỏ nhất.

**Họ bài cùng pattern:** Minimum Cost to Hire K Workers (857), Maximum Units on a Truck (1710), Reducing Dishes (1402), Assign Cookies (455), Minimum Cost to Move Chips (1217), Maximize Greatness of an Array (2592).

## 15. Biến thể

1. **Đúng `k` người đi `A`** (không nhất thiết một nửa): lấy `k` chênh lệch `a − b` nhỏ nhất.
2. **Chi phí lớn nhất** (đảo dấu): lấy `n` chênh lệch **lớn nhất**.
3. **Trả về một cách chia** cụ thể (danh sách chỉ số đi `A`).

Cài đặt và đối chiếu brute:

In [29]:
def sched_k(costs, k):
    """Biến thể 1: đúng k người đi A (0 <= k <= len)."""
    diffs = sorted(a - b for a, b in costs)
    return sum(b for a, b in costs) + sum(diffs[:k])


def sched_k_brute(costs, k):
    m = len(costs)
    return min(sum(costs[i][0] if i in set(A) else costs[i][1] for i in range(m)) for A in itertools.combinations(range(m), k))


def sched_max(costs):
    """Biến thể 2: chi phí lớn nhất khi đúng n người đi mỗi thành phố."""
    n = len(costs) // 2
    diffs = sorted((a - b for a, b in costs), reverse=True)
    return sum(b for a, b in costs) + sum(diffs[:n])


def sched_max_brute(costs):
    m, n = len(costs), len(costs) // 2
    return max(sum(costs[i][0] if i in set(A) else costs[i][1] for i in range(m)) for A in itertools.combinations(range(m), n))


rng = random.Random(15)
for _ in range(400):
    m = rng.choice([2, 4, 6, 8])
    c = [[rng.randint(1, 20), rng.randint(1, 20)] for _ in range(m)]
    k = rng.randint(0, m)
    assert sched_k(c, k) == sched_k_brute(c, k)                                           # (1)
    assert sched_max(c) == sched_max_brute(c)                                            # (2)
    A = assignment(c)
    assert len(A) == m // 2                                                              # (3)
assert sched_k([[10, 20], [30, 200], [400, 50], [30, 20]], 2) == 110
print("đúng k người đi A, chi phí lớn nhất, cách chia cụ thể: khớp brute")

đúng k người đi A, chi phí lớn nhất, cách chia cụ thể: khớp brute


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi tổ hợp | định nghĩa | `O(C(2n,n)·n)` | `O(n)` | oracle |
| 2. Sắp xếp theo `a − b` | chọn `n` chênh lệch nhỏ nhất | `O(n log n)` | `O(n)` | **đáp án chính** |
| 3. DP `dp[i][j]` | số người đi `A` | `O(n²)` | `O(n²)` | oracle độc lập, ràng buộc phức tạp |
| 4. `heapq.nsmallest` | gửi hết về `B` rồi đổi `n` người | `O(n log n)` | `O(n)` | không sắp xếp toàn bộ |
| 5. Hoán đổi cục bộ | cực tiểu địa phương = toàn cục | `O(n²·s)` | `O(n)` | minh hoạ bất biến |

Mọi cách khớp oracle tổ hợp trên toàn bộ mảng nhỏ và khớp DP với `2n` tới `100`.

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [30]:
print(Solution().twoCitySchedCost([[10, 20], [30, 200], [400, 50], [30, 20]]), Solution().twoCitySchedCost([[10, 20], [10, 20]]),
      Solution().twoCitySchedCost([[1, 1], [1, 1]]), Solution().twoCitySchedCost([[5, 1], [1, 5]]), Solution().twoCitySchedCost([[1, 100], [1, 100]]))

110 30 2 2 101


### Bài 2 (tính tay)

`costs = [[20, 60], [60, 20], [40, 30], [10, 90]]` (`2n = 4`). (a) Tính `a − b`. (b) Hai người đi `A`? (c) Tổng chi phí. Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện cách sắp xếp theo `a − b`:

In [31]:
def sort_todo(costs):
    cs = costs        # TODO 1: sắp xếp theo a - b (sorted(..., key=lambda c: c[0] - c[1]))
    n = len(cs) // 2
    return 0          # TODO 2: tổng a của n người đầu + tổng b của n người sau


# Khi làm xong, bỏ comment để kiểm tra:
# assert all(sort_todo([x[:] for x in c]) == w for c, w in examples)

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **DP** `dp[i][j]` (số người đã xét, số người đi `A`):

In [32]:
class MySolution:
    def twoCitySchedCost(self, costs):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().twoCitySchedCost([x[:] for x in c]) == w for c, w in examples)

### Bài 5 (mini challenge): đúng `k` người đi `A`

Cho `costs` (`m` người) và số `k`, trả về chi phí tối thiểu khi **đúng `k`** người đi `A` (và `m − k` đi `B`). Gợi ý: `Σ b + (tổng k giá trị a − b nhỏ nhất)`. Kiểm bằng brute.

In [33]:
def two_city_k(costs, k):
    pass            # TODO


# Khi làm xong:
# assert two_city_k([[10, 20], [30, 200], [400, 50], [30, 20]], 2) == 110 and two_city_k([[10, 20], [30, 200]], 0) == 220

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: ví dụ 1 cho `110`.
- Mức 2: `[[10,20],[10,20]]`: một người đi `A` (10), một đi `B` (20) ⇒ `30`; `[[1,100],[1,100]]`: `1 + 100`.
- Mức 3: `110 30 2 2 101`.

**Bài 2**

- Mức 1: `a − b = −40, 40, 10, −80`.
- Mức 2: hai nhỏ nhất: `−80` (người 4), `−40` (người 1).
- Mức 3: `A`: `10 + 20 = 30`; `B`: `20 + 30 = 50`; tổng `80`.

**Bài 3**

- Mức 1: `cs = sorted(costs, key=lambda c: c[0] - c[1])`.
- Mức 2: `n = len(cs) // 2`.
- Mức 3: `sum(c[0] for c in cs[:n]) + sum(c[1] for c in cs[n:])`.

**Bài 4**

- Mức 1: `dp[0][0] = 0`.
- Mức 2: `dp[i][j] = min(dp[i-1][j-1] + a, dp[i-1][j] + b)` với `j ≤ n`, `i − j ≤ n`.
- Mức 3: trả `dp[2n][n]`.

**Bài 5**

- Mức 1: `total_b = Σ b`.
- Mức 2: `diffs = sorted(a - b ...)`.
- Mức 3: `total_b + sum(diffs[:k])`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [34]:
vals = (Solution().twoCitySchedCost([[10, 20], [30, 200], [400, 50], [30, 20]]), Solution().twoCitySchedCost([[10, 20], [10, 20]]),
        Solution().twoCitySchedCost([[1, 1], [1, 1]]), Solution().twoCitySchedCost([[5, 1], [1, 5]]), Solution().twoCitySchedCost([[1, 100], [1, 100]]))
print(vals)
assert vals == (110, 30, 2, 2, 101)
print("Bài 1 OK")

(110, 30, 2, 2, 101)
Bài 1 OK


### Bài 2

In [35]:
c = [[20, 60], [60, 20], [40, 30], [10, 90]]
diffs = [a - b for a, b in c]
print("a - b =", diffs)
assert diffs == [-40, 40, 10, -80]
A = assignment(c)
print("đi A:", sorted(A))
assert A == {0, 3} and Solution().twoCitySchedCost(c) == 80 == sched_brute(c)
print("Bài 2 OK: tổng 80")

a - b = [-40, 40, 10, -80]
đi A: [0, 3]
Bài 2 OK: tổng 80


### Bài 3

In [36]:
def sort_answer(costs):
    cs = sorted(costs, key=lambda c: c[0] - c[1])
    n = len(cs) // 2
    return sum(c[0] for c in cs[:n]) + sum(c[1] for c in cs[n:])


assert all(sort_answer([x[:] for x in c]) == w for c, w in examples)
rng = random.Random(3)
for _ in range(300):
    m = rng.choice([2, 4, 6, 8])
    c = [[rng.randint(1, 30), rng.randint(1, 30)] for _ in range(m)]
    assert sort_answer(c) == sched_brute(c)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [37]:
class MySolutionAnswer:
    def twoCitySchedCost(self, costs):
        m = len(costs)
        n = m // 2
        INF = float("inf")
        dp = [[INF] * (n + 1) for _ in range(m + 1)]
        dp[0][0] = 0
        for i in range(1, m + 1):
            a, b = costs[i - 1]
            for j in range(min(i, n) + 1):
                if i - j > n:
                    continue
                best = dp[i - 1][j - 1] + a if j > 0 else INF
                if i - 1 >= j:
                    best = min(best, dp[i - 1][j] + b)
                dp[i][j] = best
        return dp[m][n]


assert all(MySolutionAnswer().twoCitySchedCost([x[:] for x in c]) == w for c, w in examples)
rng = random.Random(4)
for _ in range(300):
    m = rng.choice([2, 4, 6, 8])
    c = [[rng.randint(1, 30), rng.randint(1, 30)] for _ in range(m)]
    assert MySolutionAnswer().twoCitySchedCost(c) == sched_brute(c)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [38]:
def two_city_k(costs, k):
    diffs = sorted(a - b for a, b in costs)
    return sum(b for a, b in costs) + sum(diffs[:k])


assert two_city_k([[10, 20], [30, 200], [400, 50], [30, 20]], 2) == 110 and two_city_k([[10, 20], [30, 200]], 0) == 220
rng = random.Random(5)
for _ in range(300):
    m = rng.choice([2, 3, 4, 6, 7])
    c = [[rng.randint(1, 30), rng.randint(1, 30)] for _ in range(m)]
    k = rng.randint(0, m)
    assert two_city_k(c, k) == sched_k_brute(c, k)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (chia hai nhóm đều nhau với chi phí khác nhau → sắp xếp theo chênh lệch):**

```python
class Solution:
    def twoCitySchedCost(self, costs: List[List[int]]) -> int:
        costs = sorted(costs, key=lambda c: c[0] - c[1])
        n = len(costs) // 2
        return sum(c[0] for c in costs[:n]) + sum(c[1] for c in costs[n:])
```

**Checklist trước khi submit:**

- [ ] Khoá sắp xếp là `a − b` (không phải `a`, `b`, `min`, tỉ số)?
- [ ] Chiều tăng dần: `n` người **đầu** đi `A` (cộng `a`), `n` người sau đi `B` (cộng `b`)?
- [ ] `n = len(costs) // 2` (đúng `n` mỗi thành phố)?
- [ ] Không sửa đầu vào nếu không cần (dùng `sorted`)?
- [ ] Đã thử ví dụ 1–3, `[[1,100],[1,100]] → 101`, mọi `a = b`?
- [ ] Hiểu chứng minh đổi chỗ (cực tiểu địa phương là toàn cục)?
- [ ] Đúng tên `class Solution`, `twoCitySchedCost(self, costs)`, không in debug.